# E10 — training the two registered arms (the explicit presence decision)

Registration: `research/native/PREREG_E10.md`. Both arms are **the adopted
baseline's recipe from one seeded start**, so they begin with identical weights.
They differ in one thing:

| arm | presence |
|---|---|
| `e10_base` | the baseline: hidden prefix controls (`?` half the time) |
| `e10_plan` | a presence plan after the suffix, written by the model (so no control dropout) |

**Attach two inputs** (right panel → *Add Input*):
1. **`uie-corpus2`**, the data bundle, the same one E7, E8, E9 and N2a used;
2. **`uie-e10-code`**, the new code dataset.

**Settings:** GPU T4, Internet **On**. **Save Version → Save & Run All.** About
2.5 h. **A rerun skips a finished arm.** When done, download `/kaggle/working`
(both arm folders and `PROVENANCE.json`).

In [ ]:
# --- 1. locate both datasets ---------------------------------------------------
from pathlib import Path

root = Path("/kaggle/input")
data_hits = sorted(root.rglob("bundle.manifest.json")) if root.exists() else []
code_hits = sorted(root.rglob("E10_CODE_MANIFEST.json")) if root.exists() else []
if not data_hits or not code_hits:
    raise SystemExit("Attach BOTH datasets (right panel -> Add Input):\n"
                     "  uie-corpus2   -- the data, has bundle.manifest.json\n"
                     "  uie-e10-code  -- the E10 code, has E10_CODE_MANIFEST.json\n"
                     "then re-run.")
DATA, CODE = data_hits[0].parent, code_hits[0].parent
WORK = Path("/kaggle/working")
print("data:", DATA)
print("code:", CODE)

In [ ]:
# --- 2. environment: install first, then run THIS code and prove it ------------
import hashlib, importlib, json, subprocess, sys
from importlib import metadata

TLDEXTRACT = "5.3.1"          # the version the local evaluation parses with


def installed(dist):
    try:
        return metadata.version(dist)
    except metadata.PackageNotFoundError:
        return None


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


if installed("tldextract") != TLDEXTRACT:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        f"tldextract=={TLDEXTRACT}"], check=True)
    except subprocess.CalledProcessError:
        raise SystemExit("pip could not install tldextract: Settings -> Internet -> On, rerun.")
    importlib.invalidate_caches()
assert installed("tldextract") == TLDEXTRACT, installed("tldextract")

# CODE ONLY on the path: the data bundle carries an older `uie`.
sys.path.insert(0, str(CODE / "research" / "native"))
sys.path.insert(0, str(CODE))
for name in [m for m in sys.modules
             if m == "uie" or m.startswith("uie.") or m in ("e8_common", "e10_common")]:
    del sys.modules[name]
import uie
import e10_common as K
manifest = json.loads((CODE / "E10_CODE_MANIFEST.json").read_text())
for mod in (uie, K):
    assert str(CODE) in mod.__file__, f"{mod.__name__} is SHADOWED -- check sys.path"
bad = [f for f, h in manifest["files"].items() if sha256(CODE / f) != h]
bad += [t for t, h in manifest["tables"].items() if sha256(CODE / t) != h]
assert not bad, f"STOP. files differ from the manifest: {bad}"
print(f"code: commit {manifest['commit'][:12]}; {len(manifest['files'])} files and "
      f"{len(manifest['tables'])} tables verified")

import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU T4, then rerun.")
print("torch:", torch.__version__, "| gpu:", torch.cuda.get_device_name(0))

need = ["name_train", "structure_train", "phishing_name_train", "phishing_structure_train",
        "name_val", "structure_val", "phishing_name_val", "phishing_structure_val"]
present = {p.name for p in DATA.iterdir()}
missing = [s for s in need if f"{s}.jsonl" not in present and f"{s}.jsonl.gz" not in present]
if missing:
    raise SystemExit(f"STOP. missing {missing}: this is not the uie-corpus2 bundle.")
import inspect
from uie.kaggle_train import run
assert {"loss_norm", "loss_constant", "seed_init", "presence_plan"} <= set(
    inspect.signature(run).parameters)
print("data bundle: all 8 streams present; E10 flags present")
for arm in K.ARMS:
    print(f"  {arm}: {K.recipe(arm)}")

In [ ]:
# --- 3. train both arms (a finished arm is skipped) ------------------------------
import time

PROV = WORK / "PROVENANCE.json"
prov = json.loads(PROV.read_text()) if PROV.exists() else {"arms": {}}
prov.update({"registration": "research/native/PREREG_E10.md",
             "code_commit": manifest["commit"], "base_recipe": K.BASE})
for arm in K.ARMS:
    out = WORK / arm
    if (out / "train_result.json").exists() and (out / "best.pt").exists():
        print(f"{arm}: finished earlier -- skipped")
        continue
    out.mkdir(parents=True, exist_ok=True)
    print(f"\n==================== {arm} {K.ARMS[arm]} ====================", flush=True)
    t0 = time.time()
    run(DATA, out, **K.recipe(arm))
    prov["arms"][arm] = {"changes": K.ARMS[arm], "wall_minutes": round((time.time() - t0) / 60, 1)}
    PROV.write_text(json.dumps(prov, indent=1))
print("\nboth arms trained")

In [ ]:
# --- 4. read both arms back from their CHECKPOINTS and records --------------------
problems = []
res = {}
for arm in K.ARMS:
    rec = K.recipe(arm)
    ck = torch.load(WORK / arm / "best.pt", map_location="cpu", weights_only=False)
    tc, mc, tk = ck["train_config"], ck["model_config"], ck["tokenizer"]
    r = json.loads((WORK / arm / "train_result.json").read_text())
    res[arm] = r
    want = [("steps", tc.get("steps"), 20000), ("batch_size", tc.get("batch_size"), 256),
            ("lr", tc.get("lr"), 3e-4), ("seed", tc.get("seed"), 20260803),
            ("field_dropout", tc.get("field_dropout"), rec["field_dropout"]),
            ("class_alpha", tc.get("class_alpha"), 1.0),
            ("loss_norm", tc.get("loss_norm"), "fixed"),
            ("loss_constant", round(r.get("loss_constant") or 0, 3),
             round(rec["loss_constant"], 3)),
            ("d_model", mc.get("d_model"), 384), ("n_layers", mc.get("n_layers"), 8),
            ("vocab_size", mc.get("vocab_size"), 107),
            ("include_fields", tk.get("include_fields"), True),
            ("presence_plan", tk.get("presence_plan", False), rec.get("presence_plan", False)),
            ("root_slash", tk.get("root_slash", False), False),
            ("seed_init", r.get("seed_init"), True)]
    wrong = [w for w in want if w[1] != w[2]]
    problems += [(arm,) + w for w in wrong]
    prov["arms"].setdefault(arm, {"changes": K.ARMS[arm]})
    prov["arms"][arm].update({
        "best_sha256": sha256(WORK / arm / "best.pt"),
        "final_sha256": sha256(WORK / arm / "final.pt"),
        "init_weights_sha256": r.get("init_weights_sha256"),
        "loss_constant": r.get("loss_constant"), "final_val": r.get("final"),
        "dropped_over_max_len": r.get("dropped_over_max_len"), "counts": r.get("counts")})
    print(f"{arm:<9} val {r.get('final')}  dropped {r.get('dropped_over_max_len')}  "
          f"{'OK' if not wrong else 'WRONG ' + str(wrong)}")
starts = {res[a].get("init_weights_sha256") for a in K.ARMS}
if len(starts) != 1:
    problems.append(("V-id", "initial weights differ between the arms", starts))
PROV.write_text(json.dumps(prov, indent=1))
print("\nshared start:", "IDENTICAL" if len(starts) == 1 else "DIFFERENT", starts)
print("MISMATCHES:", problems if problems else "none -- both arms are as registered")
print("Now download /kaggle/working (e10_base/, e10_plan/ and PROVENANCE.json).")